SIGNAL TEST

The goal of this section is to add a signal component to the model and get an estimate for the posterior distribution p(mu_s|data). After that we shall proceed with model comparison. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

THE CODE BELOW IS A RELOAD OF ALL THE IMPORTANT VARIABLES

In [ ]:
bins = pd.read_csv('SourceData/s2_binning_info.csv')
resp_nr = pd.read_csv('SourceData/s2_response_nr.csv')
resp_er = pd.read_csv('SourceData/s2_response_er.csv')
bg = pd.read_csv('SourceData/er_and_cevns_background.csv')
events = pd.read_csv('SourceData/events_after_cuts.csv')

In [ ]:
s2_bin_centers_log = bins['log_center_pe'].values
s2_bin_centers_lin = bins['linear_center_pe'].values
s2_bin_widths = (bins['end_pe'] - bins['start_pe']).values
s2_bin_edges = np.concatenate([bins['start_pe'].values, [bins['end_pe'].iloc[-1]]])

In [ ]:
#Exctracting information from nuclear recoil response
s2_energies = resp_nr['energy_kev'].values
bin_starts = resp_nr['energy_bin_start_kev'].values
bin_ends = resp_nr['energy_bin_end_kev'].values
dE = bin_ends - bin_starts

response_matrix_nr = resp_nr.values[:,3:] #we start from the 4th column, since the 3 previous ones are energies. The 4th column is s2_bin_000.

In [ ]:
import wimprates as wr
reference_cross_section = 1e-45  # cm^2
rate_pertonneyearkev = wr.rate_wimp_std(
    es=s2_energies, 
    mw=4, 
    sigma_nucleon=reference_cross_section)

In [ ]:
#Most of the code below was copied directly from the Public Data Release
rate_pertonneyear = rate_pertonneyearkev * dE
rate_before_cutoff = rate_pertonneyear * 0.97678 
#The authors of the paper remove events below 0.7keV
recoil_energy_cutoff_kev = 0.7
rate_after_cutoff = rate_before_cutoff.copy()

# Which bin contains the cutoff?
cutoff_bin_index = (bin_starts < recoil_energy_cutoff_kev).sum() - 1 #This counts how many bins start below 0.7 keV, then subtracts 1 to get the index

# All bins fully below 0.7 keV are removed
rate_after_cutoff[:cutoff_bin_index] = 0

# Suppress the spectrum proportionally in the bin with the cutoff
suppress_by = (
    (recoil_energy_cutoff_kev - bin_starts[cutoff_bin_index]) 
    / bin_starts[cutoff_bin_index])
assert 0 <= suppress_by <= 1

#Only keep the part above the cutoff
rate_after_cutoff[cutoff_bin_index] *= 1 - suppress_by 

In [ ]:
b_er = bg['er_background_events']
b_cevns = bg['cevns_background_events']
b_nominal = b_er + b_cevns
k_obs, _ = np.histogram(events['s2_area_pe'].values, bins=s2_bin_edges)

s_i = rate_after_cutoff @ response_matrix_nr
b_i = b_nominal.values

Now we add a signal to the model